# 🚀 PI-DeepONet-RAR: Physics-Informed DeepONet with Adaptive Refinement
## Kaggle Execution & Evaluation Notebook

Welcome to the official Kaggle execution notebook for **PI-DeepONet-RAR**.
This notebook trains and evaluates a Physics-Informed Deep Operator Network (PI-DeepONet) to predict 2D stress distributions around a circular hole in a plate under variable tension loads (**Kirsch Problem**).

### 🌟 Key Features of this Notebook
1. **Automated Setup**: Works seamlessly whether run in a cloned repo or directly in Kaggle.
2. **GPU Acceleration Check**: Verifies PyTorch CUDA configuration for high-speed training.
3. **Flexible Execution Modes**: Run all 4 experimental arms (`baseline`, `rar_collocation`, `rar_load`, `rar_combined`) in **Quick Mode (~5 mins)** for fast testing or **Full Mode** for research benchmarks.
4. **Interactive Loss & Growth Analytics**: Plots real-time training loss convergence curves and RAR points/loads adaptive growth.
5. **Analytical Validation**: Evaluates model predictions against the exact analytical Kirsch solution and computes Stress Concentration Factor (SCF) accuracy.
6. **High-Res 2D Stress Contour Maps**: Visualizes spatial stress fields ($\sigma_{xx}, \sigma_{yy}, \sigma_{xy}$) and absolute prediction error.

In [ ]:
# ========================================================
# 1. Environment Setup & GPU Verification
# ========================================================
import os
import sys
import torch

print("="*60)
print("  Environment Setup & GPU Check")
print("="*60)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device:    {device}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: CUDA accelerator not detected! For fast training, please enable GPU in Kaggle settings (Settings -> Accelerator -> GPU P100/T4).")

# Ensure dependencies are installed
!pip install -q torch numpy matplotlib scipy pyyaml pandas


In [ ]:
# ========================================================
# 2. Directory & Path Configuration
# ========================================================
# Detect Kaggle input path vs local repository
kaggle_input = "/kaggle/input/pi-deeponet-rar"
if os.path.exists(kaggle_input):
    print(f"Found Kaggle input dataset: {kaggle_input}")
    os.chdir(kaggle_input)
elif os.path.exists("kaggle_run_all.py"):
    print(f"Running in project root: {os.getcwd()}")
elif os.path.exists("../kaggle_run_all.py"):
    os.chdir("..")
    print(f"Moved to project root: {os.getcwd()}")

# Add module search paths
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("./src"))
sys.path.insert(0, os.path.abspath("./fem_baseline"))

print(f"\n✅ Active Directory: {os.getcwd()}")
print(f"Project files found: {os.listdir('.')}")


## 🏋️‍♂️ 2. Run Experiments Suite
Execute the training pipeline directly within Kaggle:
- **`QUICK_RUN = True`**: Fast test mode (~5 mins total on GPU).
- **`QUICK_RUN = False`**: Full research training mode (~1-2 hours on GPU).

In [ ]:
# ========================================================
# 3. Training Execution
# ========================================================
QUICK_RUN = True    # Set to False for full experiment training
ARMS_TO_RUN = ["baseline", "rar_collocation", "rar_load", "rar_combined"]

import subprocess

cmd = [sys.executable, "kaggle_run_all.py"]
if QUICK_RUN:
    cmd.append("--quick")
cmd.extend(["--arms"] + ARMS_TO_RUN)

print("="*60)
print(f"  Starting Experiment Runner...")
print(f"  Mode: {'Quick (~5 min test)' if QUICK_RUN else 'Full Experiment'}")
print(f"  Arms: {', '.join(ARMS_TO_RUN)}")
print("="*60 + "\n")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode == 0:
    print("\n🎉 Training and validation completed successfully!")
else:
    print(f"\n⚠️ Execution finished with exit code {process.returncode}")


## 📊 3. Convergence & Growth Comparison
Review the training loss trajectories and adaptive domain point & load function growth during RAR.

In [ ]:
# ========================================================
# 4. Display Results & Summary Plots
# ========================================================
from IPython.display import Image, display
import os

plots_dir = os.path.join(".", "results", "plots")
loss_plot = os.path.join(plots_dir, "loss_comparison.png")
growth_plot = os.path.join(plots_dir, "rar_growth.png")
scf_plot = os.path.join(plots_dir, "scf_comparison.png")

if os.path.exists(loss_plot):
    print("📈 Training Loss Comparison Across Arms:")
    display(Image(filename=loss_plot))

if os.path.exists(growth_plot):
    print("\n🌱 Adaptive Data Growth (RAR Points & Loads):")
    display(Image(filename=growth_plot))

if os.path.exists(scf_plot):
    print("\n🎯 Stress Concentration Factor (SCF) Accuracy:")
    display(Image(filename=scf_plot))


## 📋 4. Quantitative Validation Table
Relative $L_2$ error breakdown for stresses ($\sigma_{xx}, \sigma_{yy}, \sigma_{xy}$) compared to exact Kirsch solution.

In [ ]:
# ========================================================
# 5. Validation Error Table
# ========================================================
import pandas as pd
import os

val_csv = os.path.join(".", "results", "validation_results.csv")
if os.path.exists(val_csv):
    df_val = pd.read_csv(val_csv, index_col=0)
    print("=== Validation Metrics vs Exact Kirsch Solution ===")
    display(df_val.style.format({
        "l2_sxx": "{:.4e}",
        "l2_syy": "{:.4e}",
        "l2_sxy": "{:.4e}",
        "scf": "{:.4f}"
    }).highlight_min(axis=0, color="lightgreen"))
else:
    print("⚠️ Validation results file not found. Please run training cell above.")


## 🎨 5. 2D Stress Field Verification Maps
Visual comparison between PI-DeepONet predicted stress contour and theoretical solution around the circular hole.

In [ ]:
# ========================================================
# 6. High-Resolution 2D Stress Contour Plotting
# ========================================================
import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml
import os
import sys

from src.model import PIDeepONet
from src.physics import compute_stresses
from fem_baseline.analytical_kirsch import analytical_kirsch_stress

ARM_TO_PLOT = "rar_combined"  # Select arm model to plot
model_path = os.path.join(".", "results", f"{ARM_TO_PLOT}_best.pt")
if not os.path.exists(model_path):
    model_path = os.path.join(".", "results", f"{ARM_TO_PLOT}_final.pt")

if not os.path.exists(model_path):
    print(f"⚠️ Checkpoint model for {ARM_TO_PLOT} not found.")
else:
    print(f"Loading model checkpoint: {model_path}")
    config_path = os.path.join(".", "configs", f"{ARM_TO_PLOT}.yaml")
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
        
    branch_layers = config["model"]["branch_layers"]
    trunk_layers = config["model"]["trunk_layers"]
    num_sensors = branch_layers[0]
    
    model = PIDeepONet(branch_layers=branch_layers, trunk_layers=trunk_layers, num_outputs=2)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()
    
    L, R = 1.0, 0.2
    grid_res = 150
    x = np.linspace(-L, L, grid_res)
    y = np.linspace(-L, L, grid_res)
    XX, YY = np.meshgrid(x, y)
    coords_flat = np.column_stack([XX.ravel(), YY.ravel()])
    
    mask_outside = (coords_flat[:, 0]**2 + coords_flat[:, 1]**2) >= R**2
    valid_coords = coords_flat[mask_outside]
    
    T_val = 1.0
    load_tensor = torch.ones(1, num_sensors, dtype=torch.float32) * T_val
    coords_tensor = torch.tensor(valid_coords, dtype=torch.float32, requires_grad=True)
    
    pred = model(load_tensor, coords_tensor)
    u_pred = pred[0, :, 0:1]
    v_pred = pred[0, :, 1:2]
    stresses = compute_stresses(coords_tensor, u_pred, v_pred)
    
    sxx_pred_valid = stresses["sigma_xx"].detach().numpy().squeeze()
    sxx_true_valid, _, _ = analytical_kirsch_stress(valid_coords[:, 0], valid_coords[:, 1], R=R, T=T_val)
    
    def build_grid(valid_vals):
        grid = np.full(XX.shape, np.nan)
        grid[mask_outside.reshape(XX.shape)] = valid_vals
        return grid
        
    sxx_pred_grid = build_grid(sxx_pred_valid)
    sxx_true_grid = build_grid(sxx_true_valid)
    err_sxx_grid = build_grid(np.abs(sxx_pred_valid - sxx_true_valid))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    im0 = axes[0].contourf(XX, YY, sxx_pred_grid, levels=50, cmap="viridis")
    axes[0].set_title(f"PI-DeepONet ({ARM_TO_PLOT}) $\sigma_{{xx}}$")
    axes[0].set_aspect("equal")
    fig.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].contourf(XX, YY, sxx_true_grid, levels=50, cmap="viridis")
    axes[1].set_title("Exact Kirsch Solution $\sigma_{xx}$")
    axes[1].set_aspect("equal")
    fig.colorbar(im1, ax=axes[1])
    
    im2 = axes[2].contourf(XX, YY, err_sxx_grid, levels=50, cmap="magma")
    axes[2].set_title("Absolute Error $|\sigma_{xx}^{pred} - \sigma_{xx}^{true}|$")
    axes[2].set_aspect("equal")
    fig.colorbar(im2, ax=axes[2])
    
    for ax in axes:
        circle = plt.Circle((0, 0), R, color="white", zorder=10)
        ax.add_patch(circle)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        
    plt.suptitle("2D Stress Field Verification — Model vs Analytical Kirsch", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
